In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import plotly.express as px
import plotly.graph_objects as go

# ============================================================
# Config
# ============================================================
BASE_DIR = Path("./")
INPUT_FILE = BASE_DIR / "event_representation_matrix_window0_scaled.csv"

N_CLUSTERS = 3
RANDOM_STATE = 42

# ============================================================
# 1. Load
# ============================================================
df = pd.read_csv(INPUT_FILE)
df["Date"] = pd.to_datetime(df["Date"])

feature_cols = [col for col in df.columns if col != "Date"]
X = df[feature_cols].values

print(f"[INFO] N events: {len(df)}")
print(f"[INFO] Features: {feature_cols}")

# ============================================================
# 2. KMeans Clustering
# ============================================================
kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=20
)

labels = kmeans.fit_predict(X)
df["Cluster"] = labels.astype(str)

# ============================================================
# 3. PCA
# ============================================================
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

df["PC1"] = X_pca[:, 0]
df["PC2"] = X_pca[:, 1]
df["Date_str"] = df["Date"].dt.strftime("%Y-%m-%d")

# ============================================================
# 4. Event metadata
# ============================================================
event_data = [
    # 구조 변화 event
    ("2020-11-09", "구조 변화", "코로나 백신 발표로 경기회복 기대, 금/은 안전자산 가격 급락"),
    ("2024-08-05", "구조 변화", "미국 경기둔화·침체 우려로 글로벌 위험자산 매도"),
    ("2024-12-18", "구조 변화", "Fed 정책결정 대기와 금리경로 재평가"),
    ("2025-07-08", "구조 변화", "트럼프의 구리 50% 관세 방침 발표"),
    ("2025-07-31", "구조 변화", "미국 정제동을 관세 대상에서 제외"),

    # 국소 변화 event
    ("2020-10-22", "국소 변화", "미국 경기지표 개선과 달러 반등으로 금·은 약세"),
    ("2020-10-28", "국소 변화", "유럽·미국의 코로나 재확산과 봉쇄 우려"),
    ("2020-11-04", "국소 변화", "미국 대선 개표"),
    ("2021-02-01", "국소 변화", "실버 스퀴즈 발생"),
    ("2021-06-17", "국소 변화", "FOMC 이후 달러 급등과 조기 긴축 기대 강화, 금·은·구리 동시 급락"),
    ("2022-03-01", "국소 변화", "러시아-우크라이나 전쟁 초기 충격"),
    ("2022-03-07", "국소 변화", "전쟁 충격 확대, LME 금속시장 전반 영향"),
    ("2022-06-13", "국소 변화", "미국 CPI 충격 직후, 인플레이션과 긴축 우려"),
    ("2022-07-28", "국소 변화", "recession fear"),
    ("2022-11-10", "국소 변화", "미국 CPI가 예상보다 낮게 나오며 Fed 속도조절 기대"),
    ("2022-11-17", "국소 변화", "달러 반등과 Fed 추가 긴축 발언으로 금 후퇴"),
    ("2022-12-20", "국소 변화", "중국 리오프닝 기대"),
    ("2023-01-03", "국소 변화", "중국 리오프닝과 달러 약세 기대"),
    ("2023-05-11", "국소 변화", "중국 수요 둔화·미국 부채한도 불안, 생성형 AI 인프라 투자 확대 기대"),
    ("2023-10-02", "국소 변화", "파나마 First Quantum 신규 광산계약 논의 중단"),
    ("2023-10-16", "국소 변화", "geopolitical safe-haven bid"),
    ("2024-02-15", "국소 변화", "미국 인플레이션·소매판매 지표로 rate cut 기대 흔들림"),
    ("2024-04-10", "국소 변화", "중국 경기지표 개선과 원료 부족 우려로 강세"),
    ("2024-04-30", "국소 변화", "AI·데이터센터가 2030년까지 구리 수요 100만 톤 추가 전망"),
    ("2024-11-06", "국소 변화", "미국 대선에서 트럼프 승리 확정"),
    ("2025-01-13", "국소 변화", "LME가 2022 니켈 위기 후 거래 회복 확인"),
    ("2025-02-03", "국소 변화", "무역불안에 따른 금 강세, 구조적 구리 수요 테마 명확"),
    ("2025-04-03", "국소 변화", "트럼프 공격적 관세 발표로 금·은 사상 최고치, 구리는 경기둔화 우려로 약세"),
    ("2025-11-13", "국소 변화", "미국 셧다운 종료와 더 비둘기파적 금리 기대 속 금·은 상승"),
    ("2025-12-29", "국소 변화", "금·은 급등 후 변동성 확대, 구리는 글로벌 공급부족 우려로 15년 만의 최대 연간 상승폭"),

    # 단기 충격 event
    ("2022-03-02", "단기 충격", "러-우 전쟁 초기, 안전자산 확대"),
    ("2022-09-13", "단기 충격", "미국 CPI 쇼크"),
    ("2022-12-15", "단기 충격", "Fed 금리 경로 경계 속에서도 중국 리오프닝 기대"),
    ("2025-04-08", "단기 충격", "미·중 무역갈등 재격화와 관세 불안, AI·DC 구리 수요 확대"),
    ("2025-07-09", "단기 충격", "트럼프의 구리 50% 관세 방침 발표"),
    ("2025-09-02", "단기 충격", "금·은 사상 최고권, 구리는 2개월 고점"),
    ("2025-11-17", "단기 충격", "중앙은행의 지속적 금 매수 전망"),
    ("2025-11-24", "단기 충격", "UBS의 구리 전망 상향"),
]

event_df = pd.DataFrame(event_data, columns=["Date_str", "Event_Type", "Event_Desc"])
event_df["Date"] = pd.to_datetime(event_df["Date_str"])

# 원본 df와 merge
df = df.merge(
    event_df[["Date", "Event_Type", "Event_Desc"]],
    on="Date",
    how="left"
)

# 라벨용 짧은 텍스트
event_type_short = {
    "구조 변화": "구조",
    "국소 변화": "국소",
    "단기 충격": "단기"
}

df["Event_Label"] = np.where(
    df["Event_Type"].notna(),
    df["Event_Type"].map(event_type_short) + "<br>" + df["Date_str"],
    ""
)

# hover용 텍스트
df["Event_Desc"] = df["Event_Desc"].fillna("")
df["Event_Type"] = df["Event_Type"].fillna("해당 없음")

# 라벨 표시 대상
event_points = df[df["Event_Label"] != ""].copy()

print(f"[INFO] Matched labeled events in data: {len(event_points)}")

# 데이터에 없는 이벤트 확인
missing_events = event_df.loc[~event_df["Date"].isin(df["Date"]), ["Date_str", "Event_Type", "Event_Desc"]]
if len(missing_events) > 0:
    print("[INFO] Events not found in PCA input data:")
    for _, row in missing_events.iterrows():
        print(f" - {row['Date_str']} | {row['Event_Type']} | {row['Event_Desc']}")

# ============================================================
# 5. Visualization
# ============================================================
fig = px.scatter(
    df,
    x="PC1",
    y="PC2",
    color="Cluster",
    hover_data={
        "Date_str": True,
        "Cluster": True,
        "Event_Type": True,
        "Event_Desc": True,
        "PC1": ':.4f',
        "PC2": ':.4f',
        "Date": False
    },
    title="Event Clustering (PCA) with Event Labels",
    width=1000,
    height=750
)

fig.update_traces(
    marker=dict(size=10),
    selector=dict(mode="markers")
)

fig.add_trace(
    go.Scatter(
        x=event_points["PC1"],
        y=event_points["PC2"],
        mode="text",   # markers 제거
        text=event_points["Event_Label"],
        textposition="top center",
        textfont=dict(size=10),
        name="Labeled events",
        customdata=np.stack(
            [
                event_points["Date_str"],
                event_points["Event_Type"],
                event_points["Event_Desc"]
            ],
            axis=-1
        ),
        hovertemplate=(
            "Date=%{customdata[0]}<br>"
            "Type=%{customdata[1]}<br>"
            "Desc=%{customdata[2]}<br>"
            "PC1=%{x:.4f}<br>"
            "PC2=%{y:.4f}<extra></extra>"
        )
    )
)

fig.update_layout(
    xaxis_title="PC1",
    yaxis_title="PC2",
    legend_title="Cluster",
    template="plotly_white"
)

fig.show()

[INFO] N events: 38
[INFO] Features: ['Magnitude', 'Top1_DeltaS', 'Top2_DeltaS', 'Top3_DeltaS', 'Peak', 'PeakHorizon', 'HalfLife', 'DurationAboveHalf', 'AUC']
[INFO] Matched labeled events in data: 38
